# Stan Model Building Workflow

When writing a Stan model, as when writing any other computer program,\
*the fastest way to success is to go slowly.*

* Incremental development
   + Write a (simple) model
   + Fit the model to data (either simulated or observed)
   + Check the fit 

* Then modify *(stepwise)* and repeat

* Compare successive models

## Notebook Setup

In [ ]:
# import all libraries used in this notebook
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import libpysal as sa
import matplotlib
import splot as splt
import plotnine as p9
import arviz as az
%matplotlib inline

from cmdstanpy import CmdStanModel, cmdstan_path, cmdstan_version

import warnings
warnings.filterwarnings('ignore')
from utils_dataviz import *

## Base Model:  `poisson.stan`

This file is in directory `stan/poisson.stan`.

In [ ]:
poisson_model_file = os.path.join('..', 'stan', 'poisson.stan')

with open(poisson_model_file, 'r') as file:
    contents = file.read()
    print(contents)

## Model Checking and Model Comparison

The generated quantities block of the base model contains the following statements:

```stan
generated quantities {
  array[N] int y_rep;
  vector[N] log_lik;
  {
    vector[N] eta = log_E + beta0 + xs * betas;
    y_rep = max(eta) < 26 ? poisson_log_rng(eta) : rep_array(-1, N);
    for (n in 1:N) {
      log_lik[n] = poisson_log_lpmf(y[n] | eta[n]);
    }
  }
}
```

The quantities `y_rep` and `log_lik` are used run
posterior predictive checks and leave-one-out cross-validation (LOO-CV), respectively.

### The Posterior Predictive Check 

Posterior predictive checks tests how well the fitted model captures basic features of the data.
The base model, and all models in these notebooks, use the `generated quantities` block
to declare and populate vector `y_rep` ("replicated data"),
which is used for
[posterior predictive checking](https://mc-stan.org/docs/stan-users-guide/posterior-predictive-checks.html#simulating-from-the-posterior-predictive-distribution),
as described in the Stan User's Guide.

>Posterior predictive checks are a way of measuring whether a model
does a good job of capturing relevant aspects of the data, such as
means, standard deviations, and quantiles.

>The posterior predictive distribution is the distribution over new
observations given previous observations.  It's predictive in the
sense that it's predicting behavior on new data that is not part of
the training set.  It's posterior in that everything is conditioned on
observed data $y$.

>The posterior predictive distribution for replications
$y^{\textrm{rep}}$ of the original data set $y$ given model parameters
$\theta$ is defined by
$$
p(y^{\textrm{rep}} \mid y)
= \int p(y^{\textrm{rep}} \mid \theta)
       \cdot p(\theta \mid y) \, \textrm{d}\theta.
$$

This is computed by the following statements:

```stan
  array[N] int y_rep;
  {
    vector[N] eta = log_E + beta0 + xs * betas;
    y_rep = max(eta) < 26 ? poisson_log_rng(eta) : rep_array(-1, N);
  }
```

It directly parallels the model block statements of the likelihood:

```stan
  y ~ poisson_log(log_E + beta0 + xs * betas);
```

Because the poisson_log_rng function parameter must be less than $30 \log 2$,
we first compute this as local variable `eta` (to avoid output overhead), then
use Stan's ternary operator in tandem with the vectorized
[`poisson_log_rng`](https://mc-stan.org/docs/functions-reference/unbounded_discrete_distributions.html#poisson-distribution-log-parameterization) function to efficiently populate `y_rep`.

### Leave-one-out cross-validation (LOO)

Posterior predictive checks assess how well the model captures relevant aspects of the observed data.
Leave-one-out cross-validation (LOO CV) provides a measure how well the model will generalize to
new, unseen data.
The [loo package](https://mc-stan.org/loo/) provides an implementation of the algorithm presented in

* Vehtari, A., Gelman, A. & Gabry, J. Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC. Stat Comput 27, 1413–1432 (2017). https://doi.org/10.1007/s11222-016-9696-4
* Vehtari, A.,Simpson, D.P.,  Gelman, A., Yao, Y., & Gabry, J. Pareto Smoothed Importance Sampling, arxiv, 2024. https://arxiv.org/abs/1507.02646

It simulates what would happen if we refit the model while leaving out each data point, one at a time.
Since refitting the model repeatedly is too computationally expensive), loo() uses
Pareto Smoothed Importance Sampling (PSIS) to approximate this process efficiently.

The primary metric from `loo()` is the Expected Log Predictive Density (ELPD)
which is the log-probability of new data given the model.
A higher ELPD means better predictive performance.
However, the scale is relative,
so the absolute ELPD value is not as important as the difference between ELPD scores
for different models.
This can be carried out using the `loo_compare()` function.
By convention, both functions require that the Stan model outputs
the generated quantity `log_lik`, which is simply the log likelihood.
Although the `model` block computes the likelihood, this value is not saved.
The most efficient way to capture this value is to recompute it in the
`generated quantities` block, via the following statements.

```stan
  vector[N] log_lik;
  {
    vector[N] eta = log_E + beta0 + xs * betas;
    for (n in 1:N) {
      log_lik[n] = poisson_log_lpmf(y[n] | eta[n]);
    }
  }
```

## Model Fitting

### Assemble the input data

The data block of the model declares variables:

- `N` - the number of census tracts
- `y` - the array of observed outcomes - accidents per tract
- `E` - the population per tract ("exposure")
- `K` - the number of predictors
- `xs` - the N x K data matrix of predictors

The study data and GIS data have been assembled into a single GeoJSON file.
The dataset we're using is that used in the analysis published in 2019
[Bayesian Hierarchical Spatial Models: Implementing the Besag York Mollié Model in Stan](https://www.sciencedirect.com/science/article/pii/S1877584518301175).
The data consists of motor vehicle collisions in New York City,
as recorded by the NYC Department of Transportation, between the years 2005-2014,
restricted to collisions involving school age children 5-18 years of age as pedestrians.|

In [ ]:
nyc_geodata = gpd.read_file(os.path.join('..', 'data', 'nyc_study.geojson'))
print(nyc_geodata.columns)
print(nyc_geodata['BoroName'].value_counts())

The predictors from the study data are columns: `pct_pubtransit`,`med_hh_inc`, `traffic`, `frag_index`.

In [ ]:
design_vars = np.array(['pct_pubtransit','med_hh_inc', 'traffic', 'frag_index'])

design_mat = nyc_geodata[design_vars].to_numpy()
design_mat[:, 1] = np.log(design_mat[:, 1])
design_mat[:, 2] = np.log(design_mat[:, 2])

pois_data = {"N":nyc_geodata.shape[0],
             "y":nyc_geodata['count'].astype('int'),
             "E":nyc_geodata['kid_pop'].astype('int'),
             "K":4,
             "xs":design_mat }

### Instantiate the model

Creating a `CmdStanModel` object from a `.stan` file
compiles or recompiles the model, as needed.

In [ ]:
pois_mod = CmdStanModel(stan_file=poisson_model_file)

### Run the NUTS-HMC sampler

In [ ]:
pois_fit = pois_mod.sample(data=pois_data)

### Summarize the results

In [ ]:
pois_fit_summary = pois_fit.summary()
pois_fit_summary.round(2).loc[
  ['beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]']]

## Refinement: Mean-Center Predictor Data

*In theory*, much discussion and debate about doing this -
start
[here](https://www.goldsteinepi.com/blog/thewhyandwhenofcenteringcontinuouspredictorsinregressionmodeling/index.html),
follow the links and keep going.

*In practice*, ***this helps, alot!*** and is often necessary in order to fit the model.
The BRMS package centers continuous data values on zero - [discussion here](https://discourse.mc-stan.org/t/brms-input-scaling-clarification/23601/3).

Doing this requires two additions to the model

1. In the `transformed data` block, compute the mean of a data column, then subtract the mean from the column.

2. If the regression has an intercept term, in the `generated quantities` block, adjust for this by adding back the dot-product of the mean values of each column and the regression coefficient vector to it.

There are the key changes / additions to the base model 

```stan
data {
  // no change 
}
transformed data {
  // center continuous predictors 
  vector[K] means_xs;  // column means of xs before centering
  matrix[N, K] xs_centered;  // centered version of xs
  for (k in 1:K) {
    means_xs[k] = mean(xs[, k]);
    xs_centered[, k] = xs[, k] - means_xs[k];
  }
}
parameters {
  // no change 
}
model {
  y ~ poisson_log(log_E + beta0 + xs_centered * betas);   // centered data
  // priors same
}
generated quantities {
  real beta_intercept = beta0 - dot_product(means_xs, betas);  // adjust intercept
  // compute y_rep and log_lik using xs_centered (as before)
}
```

In [ ]:
pois_xc_mod = CmdStanModel(stan_file=os.path.join('..', 
  'stan', 'poisson_ctr_preds.stan'))

In [ ]:
pois_xc_fit = pois_xc_mod.sample(data=pois_data)  

In [ ]:
pois_xc_summary = pois_xc_fit.summary()
pois_xc_summary.round(2).loc[
  ['beta_intercept', 'beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]']]

How do the samples differ?

In [ ]:
print("data matrix original scales")   
pois_fit_summary.round(2).loc[
  ['beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]']]

## Predictor Data Scales

The data variables are on different scales:

| Measures                                              | Median | Min    | Mean     | Max       |
|-------------------------------------------------------|--------|--------|----------|-----------|
| Med. household income in USD, 2010-14                 | \$53,890| \$9,327 | \$58,497  | \$232,266  |
| Pct. commute by walk/cycle/public trans, 2010-14      | 73.9   | 9.7    | 69.8     | 100.0     |
| Standardized social fragmentation index               | -0.1   | -6.7   | 0.0      | 18.7      |
| Traffic Volume (AADT), 2015                           | 19,178 | 843    | 37,248   | 276,476   |

In the previous steps, the predictor variables `med_hh_inc` and `traffic` were log-transformed.
What happens if we just use the raw data values?

In [ ]:
design_vars = np.array(['pct_pubtransit','med_hh_inc', 'traffic', 'frag_index'])
design_mat_2 = nyc_geodata[design_vars].to_numpy()

pois_data_2 = {"N":nyc_geodata.shape[0],
             "y":nyc_geodata['count'].astype('int'),
             "E":nyc_geodata['kid_pop'].astype('int'),
             "K":4,
             "xs":design_mat_2 }

print("data scaled: log income, log traffic\n", pd.DataFrame(design_mat).describe())
print("\n\ndata unscaled\n", pd.DataFrame(design_mat_2).describe())

Run the base model on this data.

In [ ]:
pois_fit_2 = pois_mod.sample(data=pois_data_2)

In [ ]:
pois_summary_2 = pois_fit_2.summary()
pois_summary_2.round(2).loc[
  ['beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]']]

The model that transforms the data matrix fails for the same fundamental reason:
the regression component `xs * betas` or `xs_centered * betas` cannot be computed,
it either overflows or underflow because the predictors are on wildly different scales.
(Transforming the data fails faster because it both both `traffic` and `med_hh_inc`
include large negative values - this underflows consistently.)

In [ ]:
# uncomment next line, sampler will fail
# pois_xc_mod.sample(data=pois_data_2)

## Best Practices

**Rescale predictors**

Ensure that all columns of the design matrix (predictor variables) are on a similar scale to improve model convergence and interpretation. Large differences in scale can lead to inefficient sampling and numerical instability.

*But* remember to account for any rescaling when trying to interpret the fitted coefficients.

**Zero-center predictors**

Use the `transformed data` block to zero-center predictor variables
and then account for the resulting effect
on the location of the intercept in the `generated quantities` block.

```stan
real beta_intercept = beta0 - dot_product(means_xs, betas);
```

**Make incremental changes, save your work**

We started with an extremely simple model and thus far, have
only looked at how different transforms of the input data affect the
sampler speed and sample size.
Each variant of the model saved in its own file, with a minimally informative name.
However, having established the importance of zero-centered, properly scaled data,
we will use the `pois_xc.stan` model as the base model going forward.

## Posterior Predictive Checks

The generated quantities variable `y_rep` is used to run posterior predictive checks.
If a model captures the data well, summary statistics such as sample mean and standard deviation
of `y` and `y_rep` should have similar values.
In particular, we expect that the observed `y` values fall within the 50% central interval
of their corresponding `y_rep` sample at least 50% of the time.

In [ ]:
y_rep_pois = pois_fit.stan_variable("y_rep")
print(ppc_central_interval(y_rep_pois, pois_data['y']))

### Posterior predictive plots - `y` vs `y_rep`

To visualize the above summary we create a plot which compares the raw count data `y` to the range of replicates for that value.
First we sort the observations in ascending order; this ordering is used to lay out the values on the x-axis.
For each observation, we plot the raw count data `y` as a point (dark blue), the central 50% interval values of `y_rep` in orange, and the entire range of `y_rep` in grey.
We have written a series of helper functions in Python and R to set up this visualization,  function`ppc_y_yrep_overlay'

In [ ]:
ppc_plot = ppc_y_yrep_overlay(y_rep_pois, pois_data['y'],
                                'Poisson model PPC\ny (blue dot) vs. y_rep (orange 50% central interval, grey full extent)')
ppc_plot

## Refinement:  Add random effects

The Poisson distribution provides a single parameter $\lambda$, which is both mean and variance.
As the above plots show, the data is overdispersed - the observed variance is greater than expected.
To improve the model fit, we can add an ordinary random-effects component - this will account for per-tract heterogeneity.
(Not to get head of ourselves, but this is one component in the BYM model).

There are the key changes / additions to the base model 
 
```stan
data {
  // no change 
}
transformed data {
  // no change, (center continuous predictors)
}
parameters {
  real beta0; // intercept
  vector[K] betas; // covariates
  vector[N] theta; // heterogeneous random effects
  real<lower=0> sigma; // random effects variance 
}
model {
  y ~ poisson_log(log_E + beta0 + xs_centered * betas + theta * sigma);
  beta0 ~ std_normal();
  betas ~ std_normal();
  theta ~ std_normal();
  sigma ~ normal(0, 5);
}
generated quantities {
  // compute log_lik, y_rep 
  {
    vector[N] eta = log_E + beta0 + xs_centered * betas + theta * sigma;
    // ..
  }
}

```

In [ ]:
pois_re_mod = CmdStanModel(stan_file=os.path.join('..', 
  'stan', 'poisson_re.stan'))
pois_re_fit = pois_re_mod.sample(data=pois_data)
pois_re_summary = pois_re_fit.summary()
pois_re_summary.round(2).loc[
  ['beta_intercept', 'beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]', 'sigma']]

How do the samples differ from the base model?

In [ ]:
pois_xc_summary.round(2).loc[
  ['beta_intercept', 'beta0', 'betas[1]', 'betas[2]', 'betas[3]', 'betas[4]']]

**RE model**

In [ ]:
y_rep_re = pois_re_fit.stan_variable("y_rep")
print(ppc_central_interval(y_rep_re, pois_data['y']))

Run the PPC plots

In [ ]:
ppc_plot = ppc_y_yrep_overlay(y_rep_re, pois_data['y'],
                                'Poisson + RE model PPC\ny (blue dot) vs. y_rep (orange 50% central interval, grey full extent)')
ppc_plot

## Model Comparison

In addition to comparing the model estimates and effective sample size (per second),
we can use LOO-CV to evaluate how well the model will perform on unseen data.

In Python, we can do this using [ArviZ](https://python.arviz.org/en/stable/).

In [ ]:
## model comparison using loo from Arviz
import arviz as az
idata_pois_xc = az.from_cmdstanpy(
    pois_xc_fit,
    posterior_predictive="y_rep",
    dims={"betas": ["covariates"]},
    coords={"covariates": design_vars},
    observed_data={"y": pois_data['y']}
)
idata_pois_xc

In [ ]:
idata_pois_re = az.from_cmdstanpy(
    pois_re_fit,
    posterior_predictive="y_rep",
    dims={"betas": ["covariates"]},
    coords={"covariates": design_vars},
    observed_data={"y": pois_data['y']}
)

az.compare({"poisson":idata_pois_xc, "poisson_re":idata_pois_re})

While the Poisson + random effects model effectively accounts for the overdispersion,
it fails to distinguish between variance due to the spatial structure of the data and
purely heterogenous variance.  The following notebooks show how to account for both.